In [ ]:
import matplotlib.pyplot as plt
import numpy as np 

# generate a relative simple times of 50 values with two distinct bumps at 10 and 30
# no sin wave, just a simple line with two bumps
x = np.linspace(0, 50, 50)
y = np.zeros(50)
y[10] = 1
y[30] = 1

y += np.arange(50) * 0.01
y += np.random.normal(0, 0.1, 50)



# plot the time series
plt.plot(x, y)

# round the values to 2 decimal places
y = np.round(y, 2)
y.tolist()

In [ ]:
simple_y = np.ones(50) * 4

complex_y = np.ones(50) * 4 
complex_y[10] = 10.5
complex_y[30] = 6.9
complex_y[15] = 6.5
complex_y+=  np.arange(50) * 0.1
plt.plot(x, simple_y)
plt.plot(x, complex_y)

complex_y = np.round(complex_y, 2)
simple_y = np.round(simple_y, 2)

In [ ]:
hard_y =  np.zeros(50)
hard_y[10] = 1
hard_y[30] = 1
hard_y[33]=2
hard_y[1]=2
hard_y += np.arange(50) * 0.01
hard_y += np.random.normal(0, 0.1, 50)
plt.plot(x, hard_y)

In [ ]:
import transformers
import torch

from transformers import LlamaForCausalLM, LlamaTokenizer, PreTrainedTokenizerFast, AutoTokenizer, AutoModelForCausalLM
model_path = "/common-repos/LMs/Llama3_converted/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)






messages = [
    {"role": "system", "content": "You are a multimodal time series question answering model. You are supposed to answer questions about time series data."},
    {"role": "user", "content": f"[TIMESERIES_START]{simple_y.tolist()}[TIMESERIES_END][QUESTION_START]How many large bumps are in the time series?[QUESTION_END]"},
    {"role": "assistant", "content": "[ANSWER_START]0[ANSWER_END]"},
    # {"role": "user", "content": f"[TIMESERIES_START]{complex_y.tolist()}[TIMESERIES_END][QUESTION_START]How many large bumps are in the time series?[QUESTION_END]"},
    # {"role": "assistant", "content": "[ANSWER_START]3[ANSWER_END]"},
    {"role": "user", "content": f"[TIMESERIES_START]{y.tolist()}[TIMESERIES_END][QUESTION_START]How many large bumps are in the time series?[QUESTION_END]"},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

print(tokenizer.decode(input_ids[0], skip_special_tokens=False))



terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
    input_ids,
    max_new_tokens=256,
    eos_token_id=terminators,
    # do_sample=True,
    temperature=0.6,
    # top_p=0.9,
)
response = outputs[0][input_ids.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True))

res = tokenizer.decode(response, skip_special_tokens=True)

messages = messages + [
#     {"role": "system", "content": "You are a time series forecasting model, always only return the forecast nothing else!"},
#     {"role": "user", "content": "I have some data that I would like to forecast. Can you help me with that? 0,1,2,3,10,11,12,13,20,21,.."},
    {"role": "assistant", "content": res},
    {"role": "user", "content": f"[TIMESERIES_START]{hard_y.tolist()}[TIMESERIES_END][QUESTION_START]Can you count the number of spikes?[QUESTION_END]"},

]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    input_ids,
    max_new_tokens=256,
    eos_token_id=terminators,
    # do_sample=True,
    temperature=0.6,
    # top_p=0.9,
)
response = outputs[0][input_ids.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True))

In [ ]:
from opp_qa.data import OppQADataModule

datamodule = OppQADataModule(batch_size=2, task="count", short=True)
datamodule.prepare_data()
datamodule.setup("fit")

batch = next(iter(datamodule.train_dataloader()))

traj = batch["trajectory"]
question = batch["question"]
answer = batch["answer"]


context = [
    {"role": "system", "content": "You are a multimodal time series question answering model. You are supposed to answer questions about time series data."},
    {"role": "user", "content": f"[TIMESERIES_START]{traj[0].tolist()}[TIMESERIES_END][QUESTION_START]{question[0]}[QUESTION_END]"},
    {"role": "assistant", "content": f"[ANSWER_START]{answer[0]}[ANSWER_END]"},
]

sample_ctx = context + [
{"role": "user", "content": f"[TIMESERIES_START]{traj[1].tolist()}[TIMESERIES_END][QUESTION_START]{question[1]}[QUESTION_END]"},

]




In [7]:
from transformers import PatchTSMixerConfig, PatchTSMixerForPrediction
from transformers import Trainer, TrainingArguments
from ts_qa.qa_data import TimeQADataModule
import torch

datamodule = TimeQADataModule("time-qa", batch_size=64, task="binary", tag="main")

datamodule.prepare_data()
datamodule.setup("fit")
train_dataset = datamodule.ds_train 
# drop_feats = list(filter(lambda x: x != "trajectory", train_dataset.features.keys())) #["trajectory"].shape

# def preprocess_function(examples):
#     return {'input_ids': torch.tensor(examples['trajectory']).view(66,-1).transpose(1,0).tolist()}

# train_dataset = datamodule.ds_train.map(preprocess_function, remove_columns=drop_feats)

train_dataset_mapped = train_dataset.map(lambda x: {"trajectory": torch.tensor(x["trajectory"]).view(-1,300).tolist()})



Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Map:   0%|          | 0/47827 [00:00<?, ? examples/s]

/tmp/ipykernel_647496/961436186.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_dataset_mapped = train_dataset.map(lambda x: {"trajectory": torch.tensor(x["trajectory"]).view(-1,300).tolist()})


In [5]:
print(train_dataset[0]["action_sequence"])
print(train_dataset[0]["trajectory"].shape)
_,_,t,_ =  next(iter(datamodule.train_dataloader()))
print(t.shape)
# 22 3 300

{'start': tensor([0., 3., 6., 9.]), 'end': tensor([ 3.,  6.,  9., 12.]), 'action': ['leaning', 'swaying', 'boxing from left upwards', 'grasping an object'], 'action_sentence': ['Leaning is the last action performed by the person', 'The activity swaying can be observed', 'The person is engaging in boxing from left upwards', 'Grasping an object is the last action performed by the person']}
torch.Size([22, 3, 300])
torch.Size([64, 66, 300])


In [6]:
from lightning import LightningModule
from transformers import PatchTSMixerConfig, PatchTSMixerForPrediction, PatchTSMixerForPretraining, TrainingArguments
from torch.utils.data import DataLoader, Dataset
from transformers.models.patchtsmixer.modeling_patchtsmixer import (
    PatchTSMixerModelOutput,
    PatchTSMixerForPredictionOutput,
    PatchTSMixerForPreTrainingOutput,
) 
import lightning as L


class TransformerPretrain(LightningModule):
    def __init__(
        self,
        lr: float = 1e-3,
    ):
        super().__init__()
        self.config = PatchTSMixerConfig(context_length = 300,
                            num_input_channels=66,
            patch_length=300//12,
            patch_stride=300//12,
            return_loss=True,
            mode="mix_channel",
            head_aggregation="use_last",
            d_model=256,
            use_positional_encoding=True,
            positional_encoding_type="sincos",)

        self.time_extractor = PatchTSMixerForPretraining(self.config)
        self.lr = lr

    def training_step(self, batch, batch_idx):
        _, _, inputs, labels = batch
        inputs = inputs.transpose(1, 2)  # batch x seq x feat
        outputs: PatchTSMixerForPreTrainingOutput = self.time_extractor(
            inputs, inputs.clone()
        )
        loss = outputs.loss
        self.log("train/loss", loss, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        _, _, inputs, labels = batch
        inputs = inputs.transpose(1, 2)
        outputs: PatchTSMixerForPreTrainingOutput = self.time_extractor(
            inputs, inputs.clone()
        )
        loss = outputs.loss
        self.log("val/loss", loss, prog_bar=True, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.RAdam(self.parameters(), lr=self.lr)

transform_model = TransformerPretrain()

trainer = L.Trainer(max_epochs=2)

trainer.fit(transform_model, datamodule)



GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name           | Type                       | Params | Mode 
----------------------------------------------------------------------
0 | time_extractor | PatchTSMixerForPretraining | 1.1 M  | train
----------------------------------------------------------------------
1.1 M     Trainable params
3.1 K     Non-trainable params
1.1 M     Total params
4.301     Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=95` in the `DataLoader` to improve performance.
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=95` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=2` reached.


In [31]:
from einops import rearrange



sample = train_dataset


# trajectory = rearrange(trajectory, "1 len f -> f len")
traj = train_dataset_mapped[0:10]["trajectory"].transpose(2, 1)

out = transform_model.time_extractor(traj) # .last_hidden_state.max(1)[0]


In [39]:
from sklearn.manifold import TSNE
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from matplotlib.colors import ListedColormap

res = out.last_hidden_state.max(1)[0]

features = []
actions = []

for i in range(len(res)):
    el = res[i]
    acts = train_dataset_mapped[i]["action_sequence"]["action"]
    
    for j in range(len(acts)):
        feat_el = el[j*3:(j*3+3)].flatten()  # Flatten the 3 tokens x 256 dimensions into a single vector
        features.append(feat_el.detach().numpy())  # Convert PyTorch tensor to numpy array
        actions.append(acts[j])

# Convert the features and actions to numpy arrays
features = np.array(features)
actions = np.array(actions)

# Encode actions to integers
label_encoder = LabelEncoder()
actions_encoded = label_encoder.fit_transform(actions)

# Apply t-SNE to reduce dimensions
tsne = TSNE(n_components=2, random_state=0)
features_2d = tsne.fit_transform(features)

# Define a discrete colormap
unique_actions = np.unique(actions)
cmap = plt.cm.get_cmap('tab10', len(unique_actions))
cmap = ListedColormap(cmap.colors[:len(unique_actions)])  # Extract only the required number of colors

# Create a scatter plot with categorical colors
plt.figure(figsize=(10, 7))
scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1], c=actions_encoded, cmap=cmap)

# Add a legend with action labels
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=cmap(i), markersize=10) for i in range(len(unique_actions))]
labels = unique_actions
plt.legend(handles, labels, title='Actions')

# Add labels and title
plt.title('t-SNE of Action Embeddings')
plt.xlabel('Component 1')
plt.ylabel('Component 2')

# Show the plot
plt.show()



AttributeError: module 'matplotlib.cm' has no attribute 'get_cmap'